(Positivity-ADR)=
# Positivity and mass preserving ADR

Given the application we are interested in,i.e. adr equations describing chemical reactions, it is of interest to have a way to preserve positivity. This is due to the fact that negative concer´ntrantions of some species are not physical and lead to faulty behavior.

To exemplify the phenomena let's consider a prototype cell (circle) in which some chemical is pushed to the edge by internal forces. We will force, using the keyword `FFflux_c_bc`, zero flux at the boundary. This leads to an ill-defined problem mathematically, since we will use $\vec{b}=(0.1,0)$. Such a velocity have a non-zero trace on the boundary and is directed outward with respect to the surface normal. This should let the chemical flow out of the cell. For the sake of this experiment we force the zero-flux behavior.

Consider thus the linear advection equation

\begin{equation*}
\partial_t u + \nabla\cdot(\vec{b}u - d \nabla u)= 0
\end{equation*}

where
- $u$ is the species considered
- $d$ is the diffusion coefficient
- $\vec{b}$ is the advection field

Let's tart by creating the cell.

In [1]:
from cosmos.utils.generate_surface_meshes import generate_circle
from ngsolve.webgui import Draw

mesh, _ = generate_circle()
Draw(mesh)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

Now let's initialize the problem

In [2]:
from ngsolve import *
from cosmos.solvers.base_problem import UnsteadyProblem

dt = Parameter(0.1)
T = 3
t = Parameter(0)
simulation = UnsteadyProblem(mesh=mesh, t=t, dt = dt, T = T)

Define the solver and the coefficients

In [3]:
from cosmos.solvers.adr import ADRSolver
solver = ADRSolver()
simulation.attach_solver(solver)

from cosmos.utils.manufactured_solution_tools import gradient
from ngsolve.webgui import Draw

u0 = exp(-2*(x**2+y**2))
Draw(u0, mesh)
b = CF((0.1,0))
d = 0.001
Fflux = CF((0,0))
Fflux_c_bc = {'boundary': Fflux}

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Subsequently we add the solver to the simulation



In [4]:
solver.AddSpecie(VorB = VOL,
                advection = b,
                diffusion = d,
                u0 = u0,
                Fflux_c_bc=Fflux_c_bc)

We can finally run the simulation and plot the final result, comparing it with the exact solution

In [5]:
simulation.run()
print('Computed solution')
solver.draw_solution()

Running Simulation...: 100%|███████████████| 30/30 [00:01<00:00, 25.30it/s]

Computed solution


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

We can verify that the mass is indeed conserved, but it is clear that the concentration of the chemical has become unstable leading to negative concentrations

In [6]:
mass0 = Integrate(u0, mesh, order = 3)
mass_end = Integrate(solver.get_solution(), mesh, order = 3)

print('Relative mass error: ', (mass_end-mass0)/mass0)

Relative mass error:  -2.0485178391043055e-05


We can fix this by setting the keyword `MP` to `True`, so that the algorithm takes care to not only preserve positivity but also the total mass.

In [7]:
solver_mp = ADRSolver()
simulation.attach_solver(solver_mp)

solver_mp.AddSpecie(VorB = VOL,
                advection = b,
                diffusion = d,
                u0 = u0,
                Fflux_c_bc=Fflux_c_bc,
                MP = True)

t.Set(0.0)
simulation.run()
print('Computed solution')
solver_mp.draw_solution()

mass_end_mp = Integrate(solver_mp.get_solution(), mesh, order = 3)

print('Relative mass error: ', (mass_end_mp-mass0)/mass0)

Running Simulation...: 100%|███████████████| 30/30 [00:02<00:00, 11.93it/s]

Computed solution


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

Relative mass error:  -2.0485178390879483e-05
